# Aircraft Landing Gear Shock Absorber: The Impact-Comfort Tradeoff
## Python Simulation

This notebook demonstrates the fundamental engineering tradeoff in landing gear design:
**a single passive spring-damper cannot be simultaneously optimized for landing impact and taxiing comfort.**

---

### System Model

The landing gear is modeled as a mass-spring-damper system:

$$M_e \ddot{z}_a(t) + c \dot{z}_a(t) + K z_a(t) = F(t)$$

| Parameter | Symbol | Value | Unit |
|-----------|--------|-------|------|
| Suspended mass | $M_e$ | 5000 | kg |
| Spring stiffness | $K$ | $M_e \cdot \omega_0^2$ | N/m |
| Damping coefficient | $c$ | $2\xi\sqrt{K M_e}$ | N·s/m |
| Natural pulsation | $\omega_0$ | $2\pi \cdot 1.6 = 10$ | rad/s |
| Damping ratio | $\xi$ | variable | — |

> $\omega_0 = 1.6$ Hz is fixed by human physiology — the body tolerates vertical excitation best at this frequency.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import matplotlib as mpl

mpl.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.figsize': (10, 5)
})

# System parameters
M_e = 5000          # Suspended mass [kg]
omega_0 = 10.0      # Natural pulsation [rad/s] — dictated by human physiology (1.6 Hz)
K = M_e * omega_0**2  # Spring stiffness [N/m]

print(f"Mass M_e = {M_e} kg")
print(f"Natural pulsation ω₀ = {omega_0} rad/s (f₀ = {omega_0/(2*np.pi):.1f} Hz)")
print(f"Spring stiffness K = {K:.0f} N/m")

Mass M_e = 5000 kg
Natural pulsation ω₀ = 10.0 rad/s (f₀ = 1.6 Hz)
Spring stiffness K = 500000 N/m


---
## 1. Impact Phase: Impulse Response

At landing, the gear experiences an impulsive force $F_V(t) = n \cdot M_e \cdot g \cdot \delta(t)$.

The transfer function from force to displacement:

$$H_1(p) = \frac{Z_a(p)}{F_V(p)} = \frac{1}{M_e p^2 + c p + K}$$

We compare three damping regimes: **underdamped** ($\xi=0.3$), **critically damped** ($\xi=1.0$), and **overdamped** ($\xi=2.0$).

In [2]:
n = 2.0   # Load factor
g = 9.81  # Gravity [m/s²]
impulse_magnitude = n * M_e * g  # Impulse strength [N·s]

xi_values = [0.3, 0.7, 1.0, 1.5, 2.0]
t_impact = np.linspace(0, 1.5, 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(xi_values)))

for xi, color in zip(xi_values, colors):
    c = 2 * xi * np.sqrt(K * M_e)
    num = [1]
    den = [M_e, c, K]
    sys = signal.TransferFunction(num, den)
    _, y, _ = signal.lsim(sys, U=np.zeros_like(t_impact), T=t_impact, X0=[0, impulse_magnitude/M_e])
    
    # Alternative: use impulse response directly
    t_out, y_imp = signal.impulse(sys, T=t_impact)
    y_scaled = y_imp * impulse_magnitude
    
    ax1.plot(t_out, y_scaled * 100, color=color, linewidth=1.5, label=f'ξ = {xi}')
    ax2.plot(t_out, y_scaled * 100, color=color, linewidth=1.5, alpha=0.3)

# Highlight critical damping
xi_crit = 1.0
c_crit = 2 * xi_crit * np.sqrt(K * M_e)
sys_crit = signal.TransferFunction([1], [M_e, c_crit, K])
t_crit, y_crit = signal.impulse(sys_crit, T=t_impact)
y_crit_scaled = y_crit * impulse_magnitude
ax2.plot(t_crit, y_crit_scaled * 100, 'b-', linewidth=2.5, label=f'ξ = {xi_crit} (critical)')

ax1.set_title('Impact Response: All Damping Ratios')
ax1.set_xlabel('Time [s]')
ax1.set_ylabel('Displacement zₐ [cm]')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_title('Highlight: Critical Damping (ξ = 1.0)')
ax2.set_xlabel('Time [s]')
ax2.set_ylabel('Displacement zₐ [cm]')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle('Phase 1 — Landing Impact: Impulse Response', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("KEY INSIGHT — Impact Phase:")
print(f"• ξ = 1.0 (critical damping): fastest return to zero, no oscillation → BEST for impact")
print(f"• ξ = 0.3 (underdamped): oscillates, slow settling → dangerous bouncing")
print(f"• ξ = 2.0 (overdamped): no oscillation but slow return → large stroke needed")

KEY INSIGHT — Impact Phase:
• ξ = 1.0 (critical damping): fastest return to zero, no oscillation → BEST for impact
• ξ = 0.3 (underdamped): oscillates, slow settling → dangerous bouncing
• ξ = 2.0 (overdamped): no oscillation but slow return → large stroke needed


C:\Users\24271\AppData\Local\Temp\ipykernel_13516\271300906.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 2. Taxiing Phase: Frequency Response (Bode Diagram)

During taxiing, runway irregularities $z_p(t)$ are the input. The transfer function includes tire stiffness $k_p$:

$$H_2(p) = \frac{Z_a(p)}{Z_p(p)} = \frac{k_p + c p}{k_p + K + c p + M_e p^2}$$

This is a **second-order system with a zero** — the zero creates fundamentally different behavior from the impact transfer function.

With $k_p = 5K$ (tire much stiffer than suspension spring):

$$H_2(p) \approx \frac{1 + 0.2p}{(1 + 0.1p)^2}$$

In [3]:
k_p = 5 * K  # Tire stiffness (much stiffer than suspension)
xi_taxi_values = [0.3, 0.5, 0.7, 1.0]

fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(10, 7))

colors_taxi = plt.cm.plasma(np.linspace(0.2, 0.9, len(xi_taxi_values)))
omega = np.logspace(-1, 2, 1000)

for xi, color in zip(xi_taxi_values, colors_taxi):
    c = 2 * xi * np.sqrt(K * M_e)
    num = [c, k_p]
    den = [M_e, c, K + k_p]
    sys_taxi = signal.TransferFunction(num, den)
    w, mag, phase = signal.bode(sys_taxi, w=omega)
    
    ax_mag.semilogx(w, mag, color=color, linewidth=1.5, label=f'ξ = {xi}')
    ax_phase.semilogx(w, phase, color=color, linewidth=1.5, label=f'ξ = {xi}')

ax_mag.set_title('Bode Diagram — H₂(p): Runway Input → Cabin Displacement')
ax_mag.set_ylabel('Magnitude [dB]')
ax_mag.legend(fontsize=9, loc='lower left')
ax_mag.grid(True, alpha=0.3)
ax_mag.axhline(y=0, color='black', linewidth=0.5, linestyle='--')

# Mark the resonant zone
ax_mag.axvspan(2, 10, alpha=0.1, color='red', label='Resonance zone')
ax_mag.text(5, ax_mag.get_ylim()[1]*0.9, 'Amplification\nregion', fontsize=9, color='red', ha='center')

ax_phase.set_xlabel('Pulsation ω [rad/s]')
ax_phase.set_ylabel('Phase [°]')
ax_phase.legend(fontsize=9)
ax_phase.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("KEY INSIGHT — Taxiing Phase:")
print("• Low ξ creates a RESONANT PEAK — runway bumps are AMPLIFIED at certain speeds")
print("• High ξ reduces the peak BUT transmits more vibration at mid frequencies")
print("• This is the OPPOSITE of what impact needs (high ξ = good for impact)")

C:\Users\24271\AppData\Local\Temp\ipykernel_13516\4285172309.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


KEY INSIGHT — Taxiing Phase:
• Low ξ creates a RESONANT PEAK — runway bumps are AMPLIFIED at certain speeds
• High ξ reduces the peak BUT transmits more vibration at mid frequencies
• This is the OPPOSITE of what impact needs (high ξ = good for impact)


---
## 3. The Fundamental Conflict

The same damping ratio $\xi$ serves two masters with opposite demands:

| Regime | What high ξ does | What low ξ does |
|--------|-----------------|----------------|
| **Impact** | Fast energy dissipation, no bounce → **GOOD** | Oscillation, slow settling → **BAD** |
| **Taxiing** | More vibration transmitted, higher resonance amplitude → **BAD** | Better isolation at high frequency → **GOOD** |

Let's quantify this directly.

In [4]:
xi_range = np.linspace(0.2, 2.0, 100)
overshoot_impact = []
resonance_peak_taxi = []

for xi in xi_range:
    c = 2 * xi * np.sqrt(K * M_e)
    
    # Impact: overshoot percentage (from impulse response)
    sys_imp = signal.TransferFunction([1], [M_e, c, K])
    t_test, y_test = signal.impulse(sys_imp, T=np.linspace(0, 2, 2000))
    y_test_scaled = y_test * impulse_magnitude
    
    # Measure undershoot (negative displacement = bounce)
    min_val = np.min(y_test_scaled)
    max_val = np.max(y_test_scaled)
    if max_val > 0:
        undershoot_pct = abs(min_val) / max_val * 100 if min_val < 0 else 0
    else:
        undershoot_pct = 0
    overshoot_impact.append(undershoot_pct)
    
    # Taxiing: resonance peak (max gain from Bode)
    num = [c, k_p]
    den = [M_e, c, K + k_p]
    sys_taxi = signal.TransferFunction(num, den)
    w_test, mag_test, _ = signal.bode(sys_taxi, w=np.logspace(-1, 2, 1000))
    resonance_peak_taxi.append(np.max(mag_test))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(xi_range, overshoot_impact, 'b-', linewidth=2)
ax1.axvline(x=1.0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Critical damping')
ax1.set_xlabel('Damping Ratio ξ')
ax1.set_ylabel('Bounce / Undershoot [%]')
ax1.set_title('Impact Performance: Less Bounce = Better')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.annotate('GOOD for impact\n(no bounce)', xy=(1.5, 5), fontsize=10, color='green',
            ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))
ax1.annotate('BAD for impact\n(bouncing)', xy=(0.35, 50), fontsize=10, color='darkred',
            ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='lightcoral', alpha=0.7))

ax2.plot(xi_range, resonance_peak_taxi, 'r-', linewidth=2)
ax2.set_xlabel('Damping Ratio ξ')
ax2.set_ylabel('Max Bode Gain [dB]')
ax2.set_title('Taxiing Performance: Lower Gain = Better')
ax2.grid(True, alpha=0.3)
ax2.annotate('GOOD for taxiing\n(low amplification)', xy=(0.3, 5), fontsize=10, color='green',
            ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))
ax2.annotate('BAD for taxiing\n(high amplification)', xy=(1.2, 10), fontsize=10, color='darkred',
            ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='lightcoral', alpha=0.7))

fig.suptitle('The Core Tradeoff: One Parameter, Two Conflicting Objectives',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("="*60)
print("THE FUNDAMENTAL CONFLICT:")
print(f"• Impact wants HIGH ξ → less bounce, faster energy dissipation")
print(f"• Taxiing wants LOW ξ → less vibration amplification")
print(f"• A passive damper with FIXED ξ CANNOT optimize both")
print(f"• This is why semi-active/active dampers exist: different ξ in different phases")
print("="*60)

THE FUNDAMENTAL CONFLICT:
• Impact wants HIGH ξ → less bounce, faster energy dissipation
• Taxiing wants LOW ξ → less vibration amplification
• A passive damper with FIXED ξ CANNOT optimize both
• This is why semi-active/active dampers exist: different ξ in different phases


C:\Users\24271\AppData\Local\Temp\ipykernel_13516\3129386992.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Passenger Comfort: Acceleration Analysis

The real constraint on taxiing comfort is **passenger acceleration** $\ddot{z}_a(t)$. The standard NF E90-401-2 defines human tolerance to vertical vibration, with peak sensitivity in the **4–8 Hz** band.

The acceleration transfer function includes an additional filter $H_3(p)$:

$$\frac{\ddot{Z}_a(p)}{Z_p(p)} = p^2 \cdot H_2(p) \cdot H_3(p)$$

where $H_3(p) = \frac{1}{1 + \frac{2\xi_1}{\omega_1}p + \frac{1}{\omega_1^2}p^2}$ with $\omega_1 = 300$ rad/s, $\xi_1 = 0.1$.

In [5]:
# Acceleration transfer function parameters
omega_1 = 300.0   # rad/s
xi_1 = 0.1

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

xi_comfort_values = [0.3, 0.5, 0.7, 1.0]
colors_comfort = plt.cm.viridis(np.linspace(0.2, 0.9, len(xi_comfort_values)))
omega_acc = np.logspace(-1, 3, 2000)

# NF E90-401-2 comfort thresholds (approximate, for illustration)
freq_hz = np.array([1, 2, 4, 8, 16, 30, 80])
accel_limit_30min = np.array([0.5, 0.4, 0.3, 0.3, 0.5, 0.8, 1.5])  # m/s², approximate

for xi, color in zip(xi_comfort_values, colors_comfort):
    c = 2 * xi * np.sqrt(K * M_e)
    
    # H₂(p): displacement transfer
    num_h2 = [c, k_p]
    den_h2 = [M_e, c, K + k_p]
    sys_h2 = signal.TransferFunction(num_h2, den_h2)
    
    # H₃(p): acceleration filter
    num_h3 = [1]
    den_h3 = [1/omega_1**2, 2*xi_1/omega_1, 1]
    sys_h3 = signal.TransferFunction(num_h3, den_h3)
    
    # p²·H₂(p)·H₃(p) for acceleration gain
    # p² in frequency domain = (jω)² = -ω², so magnitude = ω²
    w, mag_h2, _ = signal.bode(sys_h2, w=omega_acc)
    w, mag_h3, _ = signal.bode(sys_h3, w=omega_acc)
    mag_acc = 20 * np.log10(omega_acc**2) + mag_h2 + mag_h3
    
    # For a Z_p0 = 1 cm input, acceleration magnitude:
    Z_p0 = 0.01  # 1 cm runway irregularity
    accel_linear = omega_acc**2 * 10**(mag_h2/20) * 10**(mag_h3/20) * Z_p0
    
    ax.semilogx(omega_acc / (2*np.pi), accel_linear, color=color, linewidth=1.5,
               label=f'ξ = {xi}')

# NF E90-401-2 threshold (Zone A — motion sickness, 30 min exposure)
ax.fill_between([4, 8], 0, 3, alpha=0.1, color='red', label='4–8 Hz: peak human sensitivity')
ax.axhline(y=0.3, color='red', linestyle='--', linewidth=1, alpha=0.6,
          label='~0.3 m/s² discomfort threshold (30 min)')

ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Acceleration amplitude [m/s²]')
ax.set_title('Passenger Acceleration vs. Frequency (Z_p₀ = 1 cm runway input)')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim([0.5, 50])
ax.set_ylim([0, 2.5])

plt.tight_layout()
plt.show()

print("PASSENGER COMFORT ANALYSIS:")
print(f"• The 4–8 Hz band is where humans are most sensitive to vertical vibration")
print(f"• At ξ = 1.0 (critical — good for impact), the acceleration in this band exceeds comfort thresholds")
print(f"• At ξ = 0.3 (underdamped), peak acceleration is LOWER in the sensitive band but higher at resonance")
print(f"• → Neither damping value is ideal across the full frequency range")
print(f"• → This is the mathematical demonstration of the tradeoff")

PASSENGER COMFORT ANALYSIS:
• The 4–8 Hz band is where humans are most sensitive to vertical vibration
• At ξ = 1.0 (critical — good for impact), the acceleration in this band exceeds comfort thresholds
• At ξ = 0.3 (underdamped), peak acceleration is LOWER in the sensitive band but higher at resonance
• → Neither damping value is ideal across the full frequency range
• → This is the mathematical demonstration of the tradeoff


C:\Users\24271\AppData\Local\Temp\ipykernel_13516\877851719.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Interactive Parameter Sweep

Slide through damping ratios to see how the impact response and taxiing frequency response change simultaneously — in opposite directions.

In [6]:
xi_sweep = np.arange(0.2, 2.05, 0.1)
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for idx, xi in enumerate(xi_sweep):
    if idx >= 12:
        break
    ax = axes[idx]
    c = 2 * xi * np.sqrt(K * M_e)
    
    # Impact response
    sys_imp = signal.TransferFunction([1], [M_e, c, K])
    t_i, y_i = signal.impulse(sys_imp, T=np.linspace(0, 1.2, 500))
    y_i = y_i * impulse_magnitude * 100  # cm
    
    # Taxiing Bode
    num = [c, k_p]
    den = [M_e, c, K + k_p]
    sys_taxi = signal.TransferFunction(num, den)
    
    color = 'steelblue' if xi <= 0.7 else ('darkorange' if xi <= 1.3 else 'crimson')
    label_text = 'UNDERDAMPED' if xi < 0.9 else ('CRITICAL' if xi < 1.1 else 'OVERDAMPED')
    
    ax.plot(t_i, y_i, color=color, linewidth=1.5)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.set_title(f'ξ = {xi:.1f} ({label_text})', fontsize=9, fontweight='bold', color=color)
    ax.set_xlim([0, 1.2])
    ax.set_ylim([-15, 25])
    if idx >= 8:
        ax.set_xlabel('t [s]', fontsize=8)
    if idx % 4 == 0:
        ax.set_ylabel('zₐ [cm]', fontsize=8)

fig.suptitle('Impact Response vs. Damping Ratio ξ\n'
             'Left (ξ < 0.9): Good for comfort, bad for impact  |  '
             'Center (ξ ≈ 1.0): Best for impact  |  '
             'Right (ξ > 1.1): Slow recovery, large stroke',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

C:\Users\24271\AppData\Local\Temp\ipykernel_13516\3791155044.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Summary: Engineering Implications

### The Pareto Front

```
                    GOOD for impact
                         ↑
                         |     ○ ξ=1.0 (critical — ideal for impact)
                         |        \
  Impact Performance     |         \  Pareto front
  (less bounce)          |          \
                         |           ○ ξ=0.5 (compromise)
                         |              \
                         |               ○ ξ=0.3 (best for comfort)
                         |
                         └────────────────────→ GOOD for taxiing
                           Taxiing Performance (less vibration)
```

### What real engineers do about it

| Solution | How it works | Example |
|----------|-------------|--------|
| **Multi-stage orifice** | Different damping at different stroke positions | Most airliners |
| **Semi-active (MR fluid)** | Electronically adjustable damping via magnetic field | Boeing 787 |
| **Active suspension** | Powered actuator with real-time control loop | R&D / military |

### Key takeaway

> A landing gear shock absorber is not a "bad" design because it can't do both — it's a **fundamentally constrained system** where physics and human physiology dictate the compromise. The engineer's job is to understand the Pareto front and choose the operating point that best serves the mission.

For a **sales engineer**, this means: when a customer asks "why is the ride so stiff on landing?" or "why does it vibrate at certain taxi speeds?", you can explain it's not a defect — it's the necessary consequence of a deliberate design choice that prioritized safety.